# Phase 5 — Data Splitting and Median Baseline

This notebook creates the fixed 70% training, 15% validation, and 15% test split and trains the required `DummyRegressor(strategy="median")` baseline.

The Phase 4 test holdout is preserved exactly. Phase 5 does **not** generate test predictions or calculate test metrics. Every advanced model will use this same split for a fair comparison.

## 1. Imports and project paths

In [ ]:
from pathlib import Path
import sys

import joblib
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.baseline import (
    PRICE_BAND_TABLE_FILENAME, create_final_split_assignment,
    load_phase4_holdout, run_baseline,
    summarize_validation_by_price_band, train_and_validate_dummy,
)
from src.config import (
    BASELINE_MODEL_PATH, BASELINE_REPORT_PATH, BASELINE_SUMMARY_PATH,
    HOLDOUT_ASSIGNMENT_PATH, PROCESSED_DATA_PATH,
    SPLIT_ASSIGNMENT_PATH, TABLES_DIR,
)
from src.features import BASE_FEATURE_COLUMNS, load_cleaned_features
from src.validate_data import file_sha256

print(f"Project root: {PROJECT_ROOT}")
print(f"Cleaned input: {PROCESSED_DATA_PATH}")
print(f"Phase 4 holdout: {HOLDOUT_ASSIGNMENT_PATH}")

## 2. Load the cleaned data and Phase 4 holdout

The hashes identify the exact files used. The Phase 4 assignment contains row IDs, development/test labels, and the fixed price band—no predictions or performance metrics.

In [ ]:
cleaned_hash_before = file_sha256(PROCESSED_DATA_PATH)
holdout_hash_before = file_sha256(HOLDOUT_ASSIGNMENT_PATH)
cars = load_cleaned_features(PROCESSED_DATA_PATH)
phase4_holdout = load_phase4_holdout(HOLDOUT_ASSIGNMENT_PATH, expected_rows=len(cars))

print(f"Cleaned shape: {cars.shape}")
print(f"Development rows: {int(phase4_holdout['split'].eq('development').sum()):,}")
print(f"Reserved test rows: {int(phase4_holdout['split'].eq('test').sum()):,}")
print(f"Cleaned SHA-256: {cleaned_hash_before}")
print(f"Holdout SHA-256: {holdout_hash_before}")
display(phase4_holdout.head())

## 3. Create the fixed 70/15/15 split

Only Phase 4 development rows are divided into training and validation. Stratification uses the same four fixed price bands, helping each split retain rare expensive listings.

In [ ]:
split_assignment = create_final_split_assignment(phase4_holdout)
split_counts = split_assignment['split'].value_counts().rename('rows').to_frame()
split_counts['percentage'] = split_counts['rows'] / len(split_assignment) * 100
split_by_band = pd.crosstab(
    split_assignment['price_band'],
    split_assignment['split'],
).reindex(columns=['train', 'validation', 'test'])

display(split_counts)
display(split_by_band)

phase4_test_rows = set(phase4_holdout.loc[phase4_holdout['split'].eq('test'), 'row_index'])
phase5_test_rows = set(split_assignment.loc[split_assignment['split'].eq('test'), 'row_index'])
assert phase4_test_rows == phase5_test_rows
print("The Phase 4 test holdout is preserved exactly.")

## 4. Fit the training-only median baseline

The dummy model ignores car characteristics and learns only the median training price. Because a median is unchanged by a monotonic log transformation after inversion, fitting this constant baseline directly in rupees is clear and equivalent for its purpose.

We evaluate it on validation rows only.

In [ ]:
dummy_model, validation_metrics, validation_results, validation_predictions = (
    train_and_validate_dummy(cars, split_assignment)
)

metrics_table = pd.Series(validation_metrics, name='value').to_frame()
display(metrics_table)
print(f"The model predicts ₹{validation_metrics['prediction_inr']:,.0f} for every validation car.")
print(f"Validation predictions generated: {len(validation_predictions):,}")
print("Test predictions generated: 0")

## 5. Understand performance by price band

A single ₹5.60 lakh prediction cannot work equally well across the market. Positive mean error means overprediction; negative mean error means underprediction.

In [ ]:
validation_by_band = summarize_validation_by_price_band(validation_results)
display(validation_by_band)

print("Main baseline limitation:")
print("- It overprices many cars below ₹5 lakh.")
print("- It underprices higher-value cars, especially those above ₹20 lakh.")

## 6. Save and validate all Phase 5 outputs

The reusable workflow saves the split assignment, validation metrics, price-band table, charts, report, and the validated dummy artifact. It reloads the artifact and confirms that its prediction remains unchanged.

In [ ]:
baseline_summary = run_baseline()
reloaded_model = joblib.load(BASELINE_MODEL_PATH)

print(f"Split assignment: {SPLIT_ASSIGNMENT_PATH}")
print(f"Baseline summary: {BASELINE_SUMMARY_PATH}")
print(f"Baseline report: {BASELINE_REPORT_PATH}")
print(f"Saved model: {BASELINE_MODEL_PATH}")
print(f"Reloaded prediction: ₹{reloaded_model.predict(cars[list(BASE_FEATURE_COLUMNS)].head(1))[0]:,.0f}")

## 7. Review the generated charts

### Split distribution

![Fixed split distribution](../reports/figures/12_split_distribution.png)

### Dummy baseline validation

![Dummy baseline validation](../reports/figures/13_dummy_baseline_validation.png)

## 8. Final verification

Phase 6 models must beat this validation baseline and use the same split. The test set stays unavailable until final evaluation.

In [ ]:
assert file_sha256(PROCESSED_DATA_PATH) == cleaned_hash_before
assert file_sha256(HOLDOUT_ASSIGNMENT_PATH) == holdout_hash_before
assert baseline_summary['split']['counts']['train']['rows'] == 10_670
assert baseline_summary['split']['counts']['validation']['rows'] == 2_287
assert baseline_summary['split']['counts']['test']['rows'] == 2_287
assert baseline_summary['split']['test_rows_preserved_exactly'] is True
assert baseline_summary['test_set']['evaluated'] is False
assert baseline_summary['test_set']['predictions_generated'] is False
assert baseline_summary['baseline']['prediction_inr'] == 560_000.0
assert BASELINE_MODEL_PATH.exists()
assert (TABLES_DIR / PRICE_BAND_TABLE_FILENAME).exists()

print("Phase 5 verification passed.")
print("Next: Phase 6 will train untuned CatBoost, LightGBM, and XGBoost models.")